# Top-3 trials de la BO, cada uno validado con N semillas (adaptado de z601_clases)

Hasta acá se validaba con 10 semillas solo el **1 ganador** de la BO. Acá se toman los **3 mejores** trials (no solo el primero) y se valida CADA UNO con varias semillas -- para confirmar si el que ganó la búsqueda interna sigue siendo el mejor una vez que se promedia el ruido, o si alguno de los otros 2 lo supera en la validación real.

Usa el `BO_log.txt` de la corrida HT4961 (Salsa Mágica); ajustar la ruta si el log está en otra carpeta.

### Seteo del ambiente — versión LOCAL

In [1]:
PARAM_carpeta_datasets <- "C:/Users/Laura/Documents/DMEyF/datasets/"
PARAM_carpeta_exp      <- "C:/Users/Laura/Documents/DMEyF/exp/"
require("data.table")
require("lightgbm")

Caricamento del pacchetto richiesto: data.table

Caricamento del pacchetto richiesto: lightgbm



## 1. Leo el BO_log.txt y me quedo con los 3 mejores trials (por el `y` interno)

In [2]:
archivo_log <- paste0(PARAM_carpeta_exp, "HT4961/BO_log.txt")  # ajustar carpeta si hace falta

bo_log <- fread(archivo_log)

setorder(bo_log, -y)
top3 <- head(bo_log, 3)

cat("Los 3 mejores trials segun el y interno de la BO:\n")
print(top3[, .(y, num_iterations, learning_rate, feature_fraction, num_leaves,
                min_data_in_leaf, scale_pos_weight, min_gain_to_split,
                lambda_l1, lambda_l2, bagging_fraction, bagging_freq)])

Los 3 mejores trials segun el y interno de la BO:
           y num_iterations learning_rate feature_fraction num_leaves
       <int>          <int>         <num>            <num>      <int>
1: 256245000           1351    0.05475002        0.5153582       1601
2: 253660000            930    0.02731257        0.1541961       1684
3: 251064000            874    0.10818140        0.1106434       1502
   min_data_in_leaf scale_pos_weight min_gain_to_split lambda_l1 lambda_l2
              <int>            <num>             <num>     <num>     <num>
1:                1        0.9712104         0.2654667  3.262270  7.540213
2:               46        0.6223848         0.2158453  1.520327  7.548036
3:                3        0.7965424         0.2186961  1.684038  7.391657
   bagging_fraction bagging_freq
              <num>        <int>
1:        0.8329823            5
2:        0.8425105            6
3:        0.8000337            4


## 2. Cargo el dataset y arm el pool undersampleado, una sola vez

In [3]:
dataset <- fread(paste0(PARAM_carpeta_datasets, "competencia_01.csv"), stringsAsFactors= TRUE)

dataset[, clase01 := ifelse(clase_ternaria %in% c("BAJA+1", "BAJA+2"), 1L, 0L)]

campos_buenos <- setdiff(colnames(dataset), c("clase_ternaria", "clase01"))
campos_buenos <- setdiff(campos_buenos, c("cprestamos_personales", "mprestamos_personales"))

dataset_train <- dataset[foto_mes == 202104]
dfuture <- dataset[foto_mes == 202106]

param_fijos <- list(
  boosting= "gbdt", objective= "binary", metric= "auc",
  first_metric_only= FALSE, boost_from_average= TRUE, feature_pre_filter= FALSE,
  force_row_wise= TRUE, verbosity= -100,
  max_depth= -1L, max_bin= 31L,
  pos_bagging_fraction= 1.0, neg_bagging_fraction= 1.0, is_unbalance= FALSE,
  drop_rate= 0.1, max_drop= 50, skip_drop= 0.5, extra_trees= FALSE
)

cortes <- seq(4000, 19000, by= 500)

## 3. Entreno cada uno de los 3 trials con N semillas, y barro los cortes en cada corrida

In [4]:
N_SEMILLAS <- 10
set.seed(585581)
semillas <- sample(100000:999999, N_SEMILLAS)

resultados_top3 <- data.table(trial = integer(), semilla = integer(), mejor_corte = integer(), TOTAL = numeric())

for (fila_i in 1:3) {

  hp_trial <- top3[fila_i]

  param_trial <- modifyList(param_fijos, list(
    num_iterations = hp_trial$num_iterations,
    learning_rate = hp_trial$learning_rate,
    feature_fraction = hp_trial$feature_fraction,
    num_leaves = hp_trial$num_leaves,
    min_data_in_leaf = hp_trial$min_data_in_leaf,
    scale_pos_weight = hp_trial$scale_pos_weight,
    min_gain_to_split = hp_trial$min_gain_to_split,
    lambda_l1 = hp_trial$lambda_l1,
    lambda_l2 = hp_trial$lambda_l2,
    bagging_fraction = hp_trial$bagging_fraction,
    bagging_freq = hp_trial$bagging_freq
  ))

  for (semilla in semillas) {

    param_trial$seed <- semilla

    set.seed(semilla, "L'Ecuyer-CMRG")

    dtrain <- lgb.Dataset(
      data= data.matrix(dataset_train[, campos_buenos, with= FALSE]),
      label= dataset_train[, clase01]
    )

    modelo <- lgb.train(data= dtrain, param= param_trial)

    prediccion <- predict(modelo, data.matrix(dfuture[, campos_buenos, with= FALSE]))
    tb_pred <- dfuture[, list(clase_ternaria)]
    tb_pred[, prob := prediccion]
    setorder(tb_pred, -prob)

    mejor_ganancia <- -Inf
    mejor_corte_local <- NA
    for (envios in cortes) {
      tb_pred[, Predicted := 0L]
      tb_pred[1:envios, Predicted := 1L]
      ganancia <- tb_pred[Predicted == 1L, sum(ifelse(clase_ternaria == "BAJA+2", 1072500, -27500))]
      if (ganancia > mejor_ganancia) {
        mejor_ganancia <- ganancia
        mejor_corte_local <- envios
      }
    }

    resultados_top3 <- rbindlist(list(resultados_top3,
      data.table(trial = fila_i, semilla = semilla, mejor_corte = mejor_corte_local, TOTAL = mejor_ganancia)))

    rm(modelo, dtrain)
    gc(full= TRUE, verbose= FALSE)
  }

  cat("Trial", fila_i, "(y interno =", hp_trial$y, ") -- terminado\n")
}

Trial 1 (y interno = 256245000 ) -- terminado
Trial 2 (y interno = 253660000 ) -- terminado
Trial 3 (y interno = 251064000 ) -- terminado


## 4. Comparo los 3 trials — ¿el que ganó la BO sigue ganando en la validación real?

In [5]:
estadisticas_top3 <- resultados_top3[, .(
  media = mean(TOTAL),
  desvio = sd(TOTAL),
  minimo = min(TOTAL),
  maximo = max(TOTAL)
), by = trial]

setorder(estadisticas_top3, -media)
print(estadisticas_top3)

cat("\n¿El trial con mejor 'y' interno de la BO es tambien el mejor en la validacion real?\n")
cat("Trial con mejor y interno:", top3[1, .(y)]$y, " (trial 1)\n")
cat("Trial con mejor media validada:", estadisticas_top3[1, trial], "\n")

   trial     media  desvio    minimo    maximo
   <int>     <num>   <num>     <num>     <num>
1:     2 439945000 3273925 435050000 443850000
2:     1 410960000 3998180 405350000 417450000
3:     3 404085000 5906919 396550000 414700000

¿El trial con mejor 'y' interno de la BO es tambien el mejor en la validacion real?
Trial con mejor y interno: 256245000  (trial 1)
Trial con mejor media validada: 2 
